# Cleveland Heart Disease — Data Cleaning

This notebook cleans the Cleveland heart disease dataset. Two columns, thal and ca, have some missing values. We just drop any row that has a missing value in either column, instead of guessing or filling in a value. After dropping those rows we have 297 patients left out of the original 303. We also change a few columns to categorical type, check for outliers, and turn the target column into a simple yes/no label for whether the patient has heart disease.

In [14]:
import os

import pandas as pd

NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, "heart+disease")

## Load the Cleveland dataset

Reads `processed.cleveland.data` (confirmed by UCI's own `WARNING` file to
be the non-corrupted file — the raw `cleveland.data` was damaged during a
node migration) into the standard 14-column layout, mapping the `?`
missing-value marker to `NaN`.

In [15]:
COLUMN_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num",
    ]

df_cleveland = pd.read_csv(
    os.path.join(DATA_DIR, "processed.cleveland.data"), header=None, names=COLUMN_NAMES, na_values="?",
    )
df_cleveland.head()





,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [16]:
df_cleveland.shape

(303, 14)

## Inspect the data

Check shape, dtypes, missing values, and duplicates before cleaning.

In [17]:
print(df_cleveland.dtypes)
print()
print("Missing values per column:")
print(df_cleveland[["ca", "thal"]].isna().sum())
print()
print("number of duplicated rows:")
print(df_cleveland.duplicated().sum())

age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
num           int64
dtype: object

Missing values per column:
ca      4
thal    2
dtype: int64

number of duplicated rows:
0


## Handle missing values

Only two columns have missing values: ca and thal. For both, we just drop the row instead of filling in a guessed value.

Why we do it this way: thal is a category code (3, 6, or 7), not a real number, so there is no sensible average to fill in — making one up would just invent a category that does not actually exist. ca is a count of blood vessels, so it looks more like an ordinary number you could average, but that number is only known when a real fluoroscopy exam actually produced a result for that patient. Filling in a guessed value, like the column mean, still means putting a number into the dataset that was never actually measured for that patient. Since we do not want to add made-up data for either column, we drop the row instead of filling anything in, for both columns, so the rule stays consistent.

None of the rows missing ca are also missing thal, so dropping both adds up to 2 + 4 = 6 rows removed in total, leaving 297 of the original 303 records.

In [18]:
df_cleveland = df_cleveland.dropna(subset=["thal", "ca"])

print("Missing values per column:")
print(df_cleveland[["ca", "thal"]].isna().sum())
print(df_cleveland.shape)


Missing values per column:
ca      0
thal    0
dtype: int64
(297, 14)


## Encode nominal features as categorical

`cp`, `restecg`, `slope`, and `thal` are nominal codes (chest pain type,
resting ECG result, ST-segment slope, thalassemia result), not continuous
magnitudes, so they're cast to pandas `category` dtype. `sex`, `fbs`, and
`exang` are also categorical (binary) but are kept as numeric 0/1 here.

In [19]:

for col in ["cp", "restecg", "slope", "thal"]:
    df_cleveland[col] = df_cleveland[col].astype("category")

## Outlier check

Per the proposal (§3.2), outliers are retained unless clearly erroneous.
This screens for physiologically impossible values (which would indicate
data-entry errors) and separately reports how many values sit outside the
IQR fences per continuous feature — for visibility only, nothing is
removed.

In [20]:


impossible = {
    "trestbps <= 0": (df_cleveland["trestbps"] <= 0).sum(),
    "chol <= 0": (df_cleveland["chol"] <= 0).sum(),
    "thalach <= 0": (df_cleveland["thalach"] <= 0).sum(),
    "age <= 0": (df_cleveland["age"] <= 0).sum(),
    "oldpeak < 0": (df_cleveland["oldpeak"] < 0).sum(),
}
print("impossible values (which means data-entry errors):")
for check, count in impossible.items():
    print(f"  {check}: {count}")

print()
print("IQR-based outlier counts (to inspect only, not removed):")
for col in ["age", "trestbps", "chol", "thalach", "oldpeak"]:
    q1, q3 = df_cleveland[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df_cleveland[col] < lower) | (df_cleveland[col] > upper)).sum()
    print(f"  {col}: {n_outliers} values outside [{lower:.1f}, {upper:.1f}]")

impossible values (which means data-entry errors):
  trestbps <= 0: 0
  chol <= 0: 0
  thalach <= 0: 0
  age <= 0: 0
  oldpeak < 0: 0

IQR-based outlier counts (to inspect only, not removed):
  age: 0 values outside [28.5, 80.5]
  trestbps: 9 values outside [90.0, 170.0]
  chol: 5 values outside [113.5, 373.5]
  thalach: 1 values outside [83.5, 215.5]
  oldpeak: 5 values outside [-2.4, 4.0]


## Inspect the target's severity levels

num is the angiographic diagnosis on a 0–4 severity scale (0 = no
significant narrowing, 1–4 = increasing degrees of narrowing across major
vessels). This checks the class counts before collapsing it to a binary
label below.

In [21]:
df_cleveland["num"].value_counts()

num
0    160
1     54
2     35
3     35
4     13
Name: count, dtype: int64

## Binarise the target

Collapses `num` into a binary disease presence/absence label: `False` (0)
= no significant narrowing, `True` (1–4 combined) = disease present.
Stored in a new `check` column; `num` itself is kept in the dataframe
rather than dropped.

In [22]:
df_cleveland["check"] = (df_cleveland["num"] > 0).astype("category")
df_cleveland["check"].value_counts()

check
False    160
True     137
Name: count, dtype: int64

## Preprocessing pipeline definition

Defines the canonical feature groupings and a reusable `get_preprocessor()` helper that the arm notebooks (e.g. `global_model_armA.ipynb`, `clinical_stratified_armB.ipynb`) import via `%run`, so every arm applies identical preprocessing rather than each redefining its own copy.

Two feature sets are defined in `FEATURE_GROUPS`: `"all"` (all 13 raw features) and `"selected"` (the pre-specified 9-feature subset used as the primary feature set from Arm A onward; see the markdown cell below for why). `get_feature_groups(feature_set)` and `get_feature_list(feature_set)` return the grouped and flat views of whichever set is asked for, and `get_preprocessor(feature_set)` builds the matching `ColumnTransformer`. The module-level `continuous`/`nominal`/`binary` names are kept for backward compatibility with Arm B and Arm C, which import them directly via `%run`, and are set to the `"selected"` group.

`ca` (number of major vessels, 0–3) is grouped with `continuous` and standard-scaled here rather than left as an unscaled passthrough count: it is a numeric count, not an unordered category, and putting it on the same scale as the other continuous measurements matters for the L1/L2-regularised models. `cp`, `slope`, and `thal` (and, in the `"all"` set, `restecg`) are the one-hot-encoded `nominal` group, with `drop='first'`: the previous full one-hot encoding produced dummy columns that were exactly collinear (they always summed to 1), which inflates coefficient variance in linear models; dropping one category per feature removes that redundancy and leaves the dropped level as the implicit reference category. `sex` and `exang` (and, in the `"all"` set, `fbs`) are already 0/1-coded and form the `binary` group, passed through unchanged.

Each nominal feature's `categories=` is also declared explicitly as its fixed, known clinical-coding domain (e.g. `restecg` only ever takes the values 0, 1, or 2), rather than left for `OneHotEncoder` to infer from whatever a given training fold happens to contain. This is not a formality: `restecg = 1` occurs only 4 times in the full 297-patient dataset, so a small training subset -- an inner cross-validation fold, or (in `clinical_stratified_armB.ipynb`) a sex-subgroup fold -- can easily contain zero of them, which would otherwise raise `ValueError: Found unknown categories` the moment a validation patient with that code is transformed. Declaring the domain explicitly encodes no information from the data (it is a fixed enumeration of valid clinical codes, not a data-driven statistic), and it guarantees every fold's encoder recognises every valid code even when that fold's training split happens not to contain it. Only the categories for nominal columns actually present in a given feature set are passed to the encoder, since e.g. `restecg` is absent from `"selected"`.

**Namespace note:** this notebook is executed via `%run` inside each arm notebook's own namespace, after that arm has already set its own configuration (e.g. `FEATURE_SET`, `K_OUTER`, `RANDOM_STATE`, `X`, `y`). Nothing in this notebook assigns any of those names, so as not to silently overwrite the arm's values.

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Two feature sets: "all" (every raw feature) and "selected" (the
# pre-specified 9-feature subset -- see markdown below for the rationale).
# DROPPED_FEATURES is exactly the set difference, kept explicit so it's
# never left implicit in a diff between the two groups.
FEATURE_GROUPS = {
    "all": {
        "continuous": ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"],
        "nominal": ["cp", "restecg", "slope", "thal"],
        "binary": ["sex", "fbs", "exang"],
    },
    "selected": {
        "continuous": ["age", "thalach", "oldpeak", "ca"],
        "nominal": ["cp", "slope", "thal"],
        "binary": ["sex", "exang"],
    },
}

DROPPED_FEATURES = ["trestbps", "chol", "restecg", "fbs"]

# Fixed, known domain of each nominal code (the Cleveland coding scheme;
# not derived from the data -- see markdown above for why this is declared
# explicitly instead of left for OneHotEncoder to infer per fold).
NOMINAL_CATEGORIES = {
    "cp": [1.0, 2.0, 3.0, 4.0],
    "restecg": [0.0, 1.0, 2.0],
    "slope": [1.0, 2.0, 3.0],
    "thal": [3.0, 6.0, 7.0],
}


def get_feature_groups(feature_set="selected"):
    """Return a fresh copy of the continuous/nominal/binary lists for
    feature_set ("all" or "selected"), so callers can't mutate the shared
    FEATURE_GROUPS definition in place."""
    groups = FEATURE_GROUPS[feature_set]
    return {name: list(cols) for name, cols in groups.items()}


def get_feature_list(feature_set="selected"):
    """Flat list of raw feature columns (continuous + nominal + binary) for feature_set."""
    groups = get_feature_groups(feature_set)
    return groups["continuous"] + groups["nominal"] + groups["binary"]


def get_preprocessor(feature_set="selected"):
    """Build the ColumnTransformer shared by every arm's modelling pipeline,
    for the given feature_set ("all" or "selected").

    - continuous -> StandardScaler().
    - nominal -> OneHotEncoder(categories=..., drop="first"): the explicit
      `categories=` (one fixed list per nominal feature, in NOMINAL_CATEGORIES)
      guarantees every valid clinical code is recognised even if a given
      training fold/subgroup doesn't happen to contain it -- only the
      categories for nominal columns in this feature_set are looked up, so
      "selected" never references restecg's categories. Dropping the first
      category per feature avoids the exact multicollinearity of full
      one-hot encoding. `handle_unknown` is left at its scikit-learn default
      ("error") since drop="first" cannot be combined with
      handle_unknown="ignore" -- the explicit categories make that default
      safe rather than a latent crash risk. `sparse_output=False` keeps the
      transformed output a dense array.
    - binary -> "passthrough", already numeric 0/1 codes.

    `verbose_feature_names_out=False` keeps `get_feature_names_out()`
    readable (e.g. "cp_2.0" rather than "nominal__cp_2.0") for downstream
    feature-importance analysis.
    """
    groups = get_feature_groups(feature_set)
    cont, nom, binc = groups["continuous"], groups["nominal"], groups["binary"]
    nominal_categories = [NOMINAL_CATEGORIES[col] for col in nom]
    return ColumnTransformer(
        transformers=[
            ("continuous", StandardScaler(), cont),
            ("nominal", OneHotEncoder(categories=nominal_categories, drop="first", sparse_output=False), nom),
            ("binary", "passthrough", binc),
        ],
        verbose_feature_names_out=False,
    )


# Backward-compatible module-level names for Arm B and Arm C, which import
# these directly via %run rather than calling get_feature_groups(). Set to
# the "selected" feature set, the project's primary feature set.
_selected_groups = get_feature_groups("selected")
continuous = _selected_groups["continuous"]
nominal = _selected_groups["nominal"]
binary = _selected_groups["binary"]

In [24]:
EXPECTED_WIDTH = {"selected": 13, "all": 18}

for feature_set, expected_width in EXPECTED_WIDTH.items():
    prep = get_preprocessor(feature_set)
    prep.fit(df_cleveland[get_feature_list(feature_set)])
    names = list(prep.get_feature_names_out())
    print(f"[{feature_set}] output feature names ({len(names)}):", names)
    assert len(names) == expected_width, (
        f"Expected {expected_width} transformed features for feature_set={feature_set!r}, got {len(names)}."
    )

print("Both feature sets produced the expected transformed width.")

[selected] output feature names (13): ['age', 'thalach', 'oldpeak', 'ca', 'cp_2.0', 'cp_3.0', 'cp_4.0', 'slope_2.0', 'slope_3.0', 'thal_6.0', 'thal_7.0', 'sex', 'exang']
[all] output feature names (18): ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca', 'cp_2.0', 'cp_3.0', 'cp_4.0', 'restecg_1.0', 'restecg_2.0', 'slope_2.0', 'slope_3.0', 'thal_6.0', 'thal_7.0', 'sex', 'fbs', 'exang']
Both feature sets produced the expected transformed width.


## Why a pre-specified feature subset

The `"selected"` feature set drops four of the 13 raw features: `trestbps`, `chol`, `restecg`, and `fbs` (`DROPPED_FEATURES` above). These four consistently showed the weakest association with the outcome in an exploratory ranking against three independent methods -- a univariate test (chi-square for categorical/binary features, ANOVA F for continuous ones), mutual information, and random-forest impurity importance. Exact statistics from that ranking (p-values, mutual-information scores, importances) are reported in `global_model_armA.ipynb`, not repeated here.

This list is **fixed in advance**, not chosen inside cross-validation: because the exploratory ranking that produced it saw every patient in the dataset, re-deriving or re-selecting features from that same ranking inside a training fold would leak information about the held-out patients. Treating the 9-feature list as a fixed specification -- decided once, outside the modelling loop, and applied identically to every fold -- avoids that leakage. It replaces the older `SelectKBest(f_classif)` toggle that some pipelines exposed, which selected features per training fold rather than fixing them in advance.

Dropping `restecg` also removes a separate, unrelated problem: `restecg = 1` occurs only 4 times across all 297 patients, so small training subsets (an inner CV fold, or a sex subgroup in `clinical_stratified_armB.ipynb`) can easily end up with zero examples of it.

The `"all"` feature set (all 13 raw features) is kept alongside `"selected"` as a sensitivity comparison, not the primary analysis -- see `global_model_armA.ipynb` for how the two are compared.

## Save the cleaned dataset

Writes the cleaned dataframe to new file `heart+disease/cleveland_clean.csv`.

In [25]:
df_cleveland.to_csv(os.path.join(DATA_DIR, "cleveland_clean.csv"), index=False)

# check the saved cleaned data

In [26]:
df = pd.read_csv(os.path.join(DATA_DIR, "cleveland_clean.csv"))


print(df[["age","trestbps","chol","thalach","oldpeak"]].describe())
print()
for c in ["cp","restecg","slope","thal","ca","sex","fbs","exang"]:
    print(c, df[c].value_counts().to_dict())

# cp: chest pain type
# trestbps: resting blood pressure
# chol: serum cholesterol
# fbs:fasting blood sugar
# restecg: resting electrocardiographic results
# thalach: maximum heart rate achieved
# exang: exercise-induced angina
# thal: thalassemia

print(df["check"].value_counts(normalize=True))

print()
print(df.groupby("sex")["check"].agg(["size", "sum", "mean"]))
#  sum=positive num

              age    trestbps        chol     thalach     oldpeak
count  297.000000  297.000000  297.000000  297.000000  297.000000
mean    54.542088  131.693603  247.350168  149.599327    1.055556
std      9.049736   17.762806   51.997583   22.941562    1.166123
min     29.000000   94.000000  126.000000   71.000000    0.000000
25%     48.000000  120.000000  211.000000  133.000000    0.000000
50%     56.000000  130.000000  243.000000  153.000000    0.800000
75%     61.000000  140.000000  276.000000  166.000000    1.600000
max     77.000000  200.000000  564.000000  202.000000    6.200000

cp {4.0: 142, 3.0: 83, 2.0: 49, 1.0: 23}
restecg {0.0: 147, 2.0: 146, 1.0: 4}
slope {1.0: 139, 2.0: 137, 3.0: 21}
thal {3.0: 164, 7.0: 115, 6.0: 18}
ca {0.0: 174, 1.0: 65, 2.0: 38, 3.0: 20}
sex {1.0: 201, 0.0: 96}
fbs {0.0: 254, 1.0: 43}
exang {0.0: 200, 1.0: 97}
check
False    0.538721
True     0.461279
Name: proportion, dtype: float64

     size  sum      mean
sex                     
0.0    96   25 